# Accessibility, and the choice nobody writes down

Somebody hands you a table. It reports that the most accessible zone in Tyne and
Wear is City Centre & Arthur's Hill, and that the least accessible of the 145 is
Chopwell & High Spen, out on the western edge of Gateshead above the Derwent
valley. Both statements are correct. Both are also correct of a second table,
built from the same cost matrix and the same decay parameter, in which Swalwell
sits 39 places higher than it did in the first one and Sunniside & Lobley Hill
sits 39 places lower.

A Hansen value is the sum of the opportunities a zone can reach, each one
discounted by what it costs to reach it. Two decisions sit inside that sentence.
The first is the decay parameter, and it gets argued about, because beta is the
thing modellers argue about. The second is what counts as an opportunity, and in
most accessibility work that crosses a local authority desk it is never stated.

So this notebook holds the cost matrix fixed and beta at 0.1185 per minute. The
one thing that moves is what is counted at the far end. Four runs, four columns,
four rankings of the same 145 zones.

Work through the sections in order, from the top of the page to the bottom.

## Where the files are

JupyterLite runs inside your browser. Nothing is installed on your machine and
you do not need administrator rights, which is why this page opens on a
locked-down work machine.

The notebook and its data arrived with the site, so there is nothing to download
and nothing to upload. Open the file browser - the panel down the left-hand
side, or the folder icon in the far-left sidebar if it is not showing - and you
will find this arrangement already in place:

```
accessibility-hansen/
    computing-accessibility.ipynb
    data/
        zones_msoa.csv
        trip_ends_msoa.csv
        opportunities_msoa.csv
        deprivation_msoa.csv
        cost_matrix_msoa.csv
```

Every path in the code below assumes it. The notebook sits at the top of the
folder and the data sits one level under it, so moving either one will break the
loading section.

**What happens to anything you change**

Because there is no server behind this, whatever you save goes into your
browser's own storage rather than onto a network drive. That has two
consequences worth taking seriously. Anything you want to keep should be
downloaded - right-click the file in the file browser and choose **Download**.
And if you clear your browsing data, or if your employer's IT policy clears it
for you, your saved work goes with it.

Do not edit the CSV files. If you want to try something out on them, duplicate
one first and work on the copy.

**Getting back to the original**

Should you change the notebook and want the version you started with, use
**Help > Clear Browser Data**. But read the warning it gives you before
confirming. It removes everything you have stored on this site, for every
notebook here, and it cannot be undone, so download anything you care about
first.

## Checking the files are where you think they are

Run the cell below before anything else. It reports what it can see, which is
faster than reading an error message later and guessing what went wrong.

In [ ]:
import os

DATA_FOLDER = "data"

expected = [
    "zones_msoa.csv",
    "trip_ends_msoa.csv",
    "opportunities_msoa.csv",
    "deprivation_msoa.csv",
    "cost_matrix_msoa.csv",
]

print("Looking in:", os.path.abspath(DATA_FOLDER))
print()

if not os.path.isdir(DATA_FOLDER):
    print("That folder does not exist yet.")
    print("Check the folder names and check where this notebook is saved.")
else:
    found = sorted(os.listdir(DATA_FOLDER))
    for name in expected:
        status = "found" if name in found else "MISSING"
        print(f"  {name:26s} {status}")

## Looking at the cost matrix before using it

Before any arithmetic, a look at the file the arithmetic will run on. This is
the check nobody runs. A cost matrix arrives from a modeller as a CSV with a
sensible file name, it gets loaded, and by the time anything looks wrong in the
output it has been treated as ground for a fortnight.

Four things to establish, all reported below: what the columns are called,
whether the file is complete, which column carries the cost used here, and what
sits on the diagonal where a zone meets itself.

The cost used here is `gc_min`, generalised cost in minutes, being travel time
plus the distance element converted at the value of time stated in the course
data notes. Because BETA further down is expressed per minute, the two agree.
That pairing is what keeps this run clear of the fault you met earlier in the
sequence, where a per-minute beta was handed a column of kilometres and returned
output that looked perfectly reasonable.

In [ ]:
import numpy as np
import pandas as pd

costs = pd.read_csv(f"{DATA_FOLDER}/cost_matrix_msoa.csv", encoding="utf-8-sig")

print("WHAT IS IN THE COST MATRIX")
print("-" * 62)
print("Columns:", ", ".join(costs.columns))
print(f"Rows: {len(costs):,}     Origins: {costs['origin_id'].nunique()}"
      f"     Destinations: {costs['destination_id'].nunique()}")
print(f"A complete 145 by 145 matrix would be {145 * 145:,} rows.")
print(f"Missing values anywhere in the file: {int(costs.isnull().sum().sum())}")
print()

same = costs[costs["origin_id"] == costs["destination_id"]]
diff = costs[costs["origin_id"] != costs["destination_id"]]

print(f"Rows where origin and destination are the same zone: {len(same)}")
print(f"  gc_min   smallest {same['gc_min'].min():6.2f}"
      f"   median {same['gc_min'].median():6.2f}"
      f"   largest {same['gc_min'].max():6.2f}")
print(f"Rows between different zones: {len(diff):,}")
print(f"  gc_min   smallest {diff['gc_min'].min():6.2f}"
      f"   median {diff['gc_min'].median():6.2f}"
      f"   largest {diff['gc_min'].max():6.2f}")

## The diagonal, and what it does to the answer

Those 145 rows where a zone meets itself carry a generalised cost between 0.85
and 8.61 minutes, median 1.83. They are not zero, and they are not real journeys
either. A zone has no road route to itself, so the intrazonal cost in this
dataset was set by rule: half the generalised cost to the nearest other zone.

The rule matters more here than it did in the gravity model. At beta 0.1185, a
cost of 1.83 minutes carries a weight of 0.805, so a zone's own opportunities
enter its accessibility sum at four-fifths of full value while a destination 20
minutes away enters at 0.094. Any zone with a large employment site inside its
own boundary scores well on that account alone.

How far does it go? The output table prints an own-zone share for every zone, and
under the jobs column the median is 3.2 per cent, small enough to leave the
ranking mostly about the network. City Centre & Arthur's Hill is the exception at
38.0 per cent, with Hetton-le-Hole North at 20.0 per cent behind it. Neither
figure invalidates anything. Both follow from a rule somebody chose, and they
belong in any note sent out with these numbers.

## Parameters

This is the only cell in the notebook you will change. Everything below it reads
these three values and does as it is told.

DECAY_FUNCTION sets the shape of the curve that turns a cost into a weight. It
is the same object as the deterrence function in the gravity model, doing a
different job, and it stays on `"exponential"` for all four runs.

BETA is the decay parameter, in units of one over a minute of generalised cost,
and it stays at 0.1185 - the value calibrated at this scale earlier in the
module. Moving beta and the opportunity variable in the same run would leave you
unable to say which of them caused what.

OPPORTUNITY names the column counted at the destination end. It takes one of
four values, as a string: `"jobs"`, `"retail_jobs"`, `"workplace_population"` or
`"population"`. You will set it to each of the four in turn.

In [ ]:
# ---------------------------------------------------------------------------
# PARAMETERS
# ---------------------------------------------------------------------------

DECAY_FUNCTION = "exponential"   # "exponential" or "power"

BETA = 0.1185            # decay parameter, per minute of generalised cost

OPPORTUNITY = "jobs"     # "jobs", "retail_jobs",
                         # "workplace_population" or "population"

# ---------------------------------------------------------------------------

## Loading the data

Five files, of which the cost matrix is much the largest at 21,025 rows, so give
the cell a moment before deciding it has stalled. The zone list fixes the order
of everything else, so that row three of the cost matrix and row three of the
opportunity table refer to the same place.

Where each column comes from: `jobs` from the trip-ends file, `retail_jobs` and
`workplace_population` from the opportunities file, and `population` from the
deprivation file. Only that one column is read out of the deprivation file. The
deprivation scores sitting beside it are for a later part of the course.

In [ ]:
zones = pd.read_csv(f"{DATA_FOLDER}/zones_msoa.csv", encoding="utf-8-sig")
trip_ends = pd.read_csv(f"{DATA_FOLDER}/trip_ends_msoa.csv",
                        encoding="utf-8-sig")
opps = pd.read_csv(f"{DATA_FOLDER}/opportunities_msoa.csv",
                   encoding="utf-8-sig")
depriv = pd.read_csv(f"{DATA_FOLDER}/deprivation_msoa.csv",
                     encoding="utf-8-sig")

zone_ids = list(zones["zone_id"])

opportunities = (zones[["zone_id", "zone_name", "local_authority"]]
                 .merge(trip_ends[["zone_id", "jobs"]], on="zone_id")
                 .merge(opps, on="zone_id")
                 .merge(depriv[["zone_id", "population"]], on="zone_id")
                 .set_index("zone_id")
                 .reindex(zone_ids))

cost = (costs
        .pivot(index="origin_id", columns="destination_id", values="gc_min")
        .reindex(index=zone_ids, columns=zone_ids)
        .values.astype(float))

CHOICES = ["jobs", "retail_jobs", "workplace_population", "population"]

print(f"Zones loaded:          {len(zone_ids)}")
print(f"Cost matrix:           {cost.shape[0]} by {cost.shape[1]}")
print(f"Opportunity columns:   {', '.join(CHOICES)}")

## What each column actually counts

Four columns, four different questions.

- `jobs` is the destination margin of the Census 2011 commuting matrix WU03EW.
  371,585 across the five authorities, being the number of recorded
  journeys-to-work ending in each zone.
- `retail_jobs` is a headcount, not a floorspace measure, whatever the word
  retail suggests elsewhere in your working life. It is workplace employment in
  SIC section G from Census 2011 WP605EW: 80,256 across the study area, of which
  Swalwell alone holds 5,944. That is 77.7 per cent of every job in that zone,
  and it is the MetroCentre.
- `workplace_population` counts 522,161, more than `jobs` because it includes
  home workers and people with no fixed workplace, counted where they live
  rather than where they work. Two official counts of apparently the same thing,
  disagreeing by 150,576, and both defensible.
- `population` is usual resident population, 1,120,521 across the 145 zones. It
  is the only column here that measures no kind of employment at all.

In [ ]:
print("THE FOUR OPPORTUNITY COLUMNS")
print("-" * 62)
for name in CHOICES:
    column = opportunities[name]
    largest = column.idxmax()
    print(f"{name:22s} total {column.sum():9,.0f}   largest zone: "
          f"{opportunities.loc[largest, 'zone_name']} ({column.max():,.0f})")
print()
print(f"Column selected for this run: {OPPORTUNITY}")

## Turning cost into weight

One line of arithmetic, and beta is the only thing in it under your control. The
four weights printed below are the whole of what the decay function does to this
study area: a pair costing under a minute keeps nine-tenths of its opportunities,
a pair at 20 minutes keeps under a tenth, and the dearest pair in Tyne and Wear
keeps almost nothing.

Compare that with the cumulative measure from the reading, where a 30-minute
threshold gives everything inside it a weight of one and everything outside it a
weight of zero. Here the cliff becomes a curve - and the arbitrary cut-off
becomes an arbitrary rate of decline, which is a different thing but not
obviously a smaller assumption.

In [ ]:
if DECAY_FUNCTION == "exponential":
    decay = np.exp(-BETA * cost)
elif DECAY_FUNCTION == "power":
    decay = cost ** (-BETA)
else:
    raise ValueError('DECAY_FUNCTION must be "exponential" or "power"')

print("THE DECAY FUNCTION")
print("-" * 62)
print(f"{DECAY_FUNCTION}, beta = {BETA} per minute of generalised cost")
print()
print(f"Weight on a pair costing {cost.min():5.2f} min: {decay.max():.4f}")
print(f"Weight on a pair costing 20.00 min: {np.exp(-BETA * 20):.4f}")
print(f"Weight on a pair costing 45.00 min: {np.exp(-BETA * 45):.4f}")
print(f"Weight on a pair costing {cost.max():5.2f} min: {decay.min():.4f}")

## The measure itself

Multiply every zone's opportunity count by the weight on the pair that reaches
it, add up across all 145 destinations, and that origin's accessibility value is
the total. One line of matrix arithmetic replaces 21,025 multiplications.

The table below reports every zone's value, its rank out of 145, and the share
of its total that comes from its own opportunities. Read the top ten and the
bottom five now. Read the own-zone column too, because it is the quickest way to
identify a zone whose position is being carried by what happens to sit inside it.

In [ ]:
supply = opportunities[OPPORTUNITY].values.astype(float)
accessibility = decay @ supply

own_share = (np.diag(decay) * supply) / accessibility

table = pd.DataFrame({
    "zone_id": zone_ids,
    "zone_name": opportunities["zone_name"].values,
    "local_authority": opportunities["local_authority"].values,
    "accessibility": accessibility,
    "own_zone_pct": own_share * 100,
})
table["rank"] = table["accessibility"].rank(ascending=False).astype(int)
table = table.sort_values("rank").reset_index(drop=True)

show = table.copy()
show["accessibility"] = show["accessibility"].map(lambda v: f"{v:,.1f}")
show["own_zone_pct"] = show["own_zone_pct"].map(lambda v: f"{v:.1f}")

print(f"ACCESSIBILITY, opportunity variable = {OPPORTUNITY}, beta = {BETA}")
print("-" * 78)
print("Ten most accessible zones")
print(show.head(10).to_string(index=False))
print()
print("Five least accessible zones")
print(show.tail(5).to_string(index=False))
print()
print(f"Own-zone share of the sum:  median {np.median(own_share) * 100:.1f}"
      f" per cent    largest {own_share.max() * 100:.1f} per cent")

## A value of what?

Take the largest number in the table you have just produced and ask what its
units are.

Under the jobs column it is 91,870.4. That is not a count of jobs: the 43,214
jobs in City Centre & Arthur's Hill were discounted by a weight of 0.9042 before
they were added to anything, and every other zone's contribution was discounted
harder. It is not minutes. It is not a percentage, or a rate, or a density. The
same zone scores 18,508.6 when retail employment is counted and 185,810.2 when
residents are counted, and none of those three numbers is more nearly correct
than the others, because none of them is a quantity of anything you could name to
a committee.

Two forms survive, and the cell below prints both. One zone against another
inside the same run, which is what a ranking is. One zone against itself in a
different scenario, which is what you will do later in the course when a scheme
changes the cost matrix. A Hansen value quoted on its own is worse than useless,
because the decimal places lend it an authority that the number does not have.

In [ ]:
top = table.iloc[0]
bottom = table.iloc[-1]

print(f"ONE VALUE, INTERROGATED    (opportunity = {OPPORTUNITY})")
print("-" * 62)
print(f"{top['zone_name']}: {top['accessibility']:,.1f}")
print(f"Raw {OPPORTUNITY} count in that same zone: "
      f"{opportunities.loc[top['zone_id'], OPPORTUNITY]:,.0f}")
print()
print("Comparative forms, which are the ones that carry meaning:")
print(f"  Highest against lowest ({bottom['zone_name']}): "
      f"{top['accessibility'] / bottom['accessibility']:.1f} to 1")
print()
print("  As a percentage of the highest-scoring zone:")
for name in ["Gateshead Town", "South Shields West", "Chopwell & High Spen"]:
    row = table[table["zone_name"] == name].iloc[0]
    print(f"    {name:24s} {row['accessibility'] / top['accessibility'] * 100:5.1f}"
          f"    rank {row['rank']:3d}")

## Does the top of the ranking look right?

Under three of the four columns the answer is the one a Tyne and Wear planner
would give without running anything. City Centre & Arthur's Hill comes first
under `jobs`, under `retail_jobs` and under `workplace_population`, and the same
five zones fill the top five each time, reordered slightly.

Under `population` it does not. Shieldfield & Heaton Park takes first place and
the city centre drops to second, which is not an error. The city centre MSOA
holds 43,214 jobs and 16,362 residents; Shieldfield & Heaton Park holds 7,743
jobs and 17,015 residents, the largest resident population of the 145. Count
people instead of jobs and the centre of gravity shifts.

The bottom of the ranking is steadier than the top. Chopwell & High Spen comes
last under all four columns and Hetton-le-Hole South is in the final three under
all four, so peripherality here is not sensitive to what you count. That is the
one part of the output you could quote without much qualification.

## Keeping the four runs beside each other

One ranking tells you nothing about the argument. The cell below writes this
run into `accessibility-by-opportunity.csv` in this folder, keyed on the
opportunity variable, so that running the same column twice replaces its
earlier columns rather than adding a second set. Once a second run exists, it
also prints the zones that moved furthest against the first run recorded.

Two numbers come out of that comparison and they say different things. The count
of zones whose rank changed at all is close to useless - small values sit very
close together in the middle of this distribution, where ranks 60 to 70 are
separated by 6.8 per cent of their own magnitude against 24.7 per cent among the
top ten, so almost any change reshuffles them. The largest single movement is the
figure that matters. Watch how far apart those two can be.

Download the CSV before you leave the page. It is what your written answer has to
refer to, and it is what you submit.

In [ ]:
RUN_RECORD = "accessibility-by-opportunity.csv"

this_run = (table[["zone_id", "zone_name", "local_authority",
                   "accessibility", "rank"]]
            .rename(columns={"accessibility": f"access_{OPPORTUNITY}",
                             "rank": f"rank_{OPPORTUNITY}"}))
this_run[f"access_{OPPORTUNITY}"] = this_run[f"access_{OPPORTUNITY}"].round(1)

if os.path.exists(RUN_RECORD):
    previous = pd.read_csv(RUN_RECORD, encoding="utf-8-sig")
    previous = previous.drop(columns=[f"access_{OPPORTUNITY}",
                                      f"rank_{OPPORTUNITY}"], errors="ignore")
    record = previous.merge(
        this_run.drop(columns=["zone_name", "local_authority"]),
        on="zone_id", how="outer")
else:
    record = this_run

order = ["zone_id", "zone_name", "local_authority"]
for name in CHOICES:
    for prefix in ("access_", "rank_"):
        if prefix + name in record.columns:
            order.append(prefix + name)

record = record[order].sort_values(f"rank_{OPPORTUNITY}").reset_index(drop=True)
record.to_csv(RUN_RECORD, index=False, encoding="utf-8-sig")

recorded = [c[5:] for c in record.columns if c.startswith("rank_")]
print("RUNS RECORDED SO FAR:", ", ".join(recorded))
print()

if len(recorded) > 1:
    first = recorded[0]
    moves = record.copy()
    moves["move"] = moves[f"rank_{first}"] - moves[f"rank_{OPPORTUNITY}"]
    columns = ["zone_name", "local_authority",
               f"rank_{first}", f"rank_{OPPORTUNITY}", "move"]

    print(f"RANK MOVEMENTS, {first} to {OPPORTUNITY}")
    print("-" * 74)
    print(f"Risen furthest under {OPPORTUNITY}")
    print(moves.nlargest(5, "move")[columns].to_string(index=False))
    print()
    print(f"Fallen furthest under {OPPORTUNITY}")
    print(moves.nsmallest(5, "move")[columns].to_string(index=False))
    print()
    changed = int((moves["move"] != 0).sum())
    print(f"Zones whose rank changed at all: {changed} of {len(moves)}")
    print(f"Largest single movement:         "
          f"{int(moves['move'].abs().max())} places")

In [ ]:
import matplotlib.pyplot as plt

if len(recorded) > 1:
    first = recorded[0]
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    ax.plot([1, 145], [1, 145], color="#9CA3AF", linewidth=1)
    ax.scatter(record[f"rank_{first}"], record[f"rank_{OPPORTUNITY}"],
               s=22, color="#C8102E", alpha=0.75, edgecolor="none")
    ax.set_xlabel(f"Rank when {first} is counted")
    ax.set_ylabel(f"Rank when {OPPORTUNITY} is counted")
    ax.set_title("Same zones, same costs, same beta")
    ax.set_xlim(0, 146)
    ax.set_ylim(0, 146)
    ax.invert_xaxis()
    ax.invert_yaxis()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()
else:
    print("Nothing to plot yet. This chart appears from the second run onwards.")

## What to do now

You have just completed the first of four runs, with OPPORTUNITY set to `"jobs"`.
Write down the top three zones and the bottom three.

Then work through the remaining three columns. For each one, change OPPORTUNITY
in the parameters cell, restart and run everything from the top using **Kernel >
Restart Kernel and Run All Cells**, and read the movement panel:

1. `"retail_jobs"` - retail employment only.
2. `"workplace_population"` - the wider Census workplace count.
3. `"population"` - usual residents, and not an employment measure at all.

Two of those three will surprise you and one will not. Note which is which
before you read the closing section, because your answer is more interesting if
it records what you expected.

When the four runs are done, download `accessibility-by-opportunity.csv` by
right-clicking it in the file browser. Your written interpretation goes with it.

## Read this once your four runs are finished

Four things the runs establish, the last of them a matter of judgement rather
than arithmetic.

Changing the opportunity variable changes the ranking, and it changes it by
enough to matter. Between `jobs` and `retail_jobs` the largest single movement is
39 places out of 145: Swalwell climbs from 67th to 28th and Whickham from 114th
to 75th, while Holystone & Benton falls from 54th to 67th. The movement panel always compares against your
first run, so the sharpest pair is one you have to look up in the exported table
yourself: set `retail_jobs` against `population` and the largest movement is 67
places, Swalwell falling from 28th back to 95th. Nothing about the network
changed. One column of the input did.

Every movement has an explanation, and it is always about what sits nearby rather
than about connectivity. Swalwell holds 5,944 retail jobs out of 7,647, so
counting retail employment lifts every zone within a few minutes of the
MetroCentre and drops the zones near a hospital or a business park. Under
`population` the direction reverses across the river: Cleadon Park, Biddick Hall
and Whiteleas in South Tyneside each rise about 20 places, South Shields being
dense in residents and thin in workplaces.

Then there is the pairing that does nothing. Between `jobs` and
`workplace_population`, 107 of the 145 zones change rank - and the largest
movement anywhere in that comparison is 9 places. Two official Census counts
differing by 150,576 people produce, for any practical purpose, the same ranking,
because the workers they disagree about are home and no-fixed-place workers
counted at their residence, and residents are spread far more evenly across these
zones than jobs are. Notice what that does to the count statistic. On the "zones
whose rank changed" measure this pair looks nearly as unstable as the retail
comparison, and it is not. The count is the wrong statistic; the largest movement
is the right one.

**Which column would I use, and why the others are worse.** For appraising a
transport scheme in this study area I would use `jobs`, and I would say so in the
first line of the note. Two reasons. It is the destination margin of the same
WU03EW matrix that supplied the trip ends and the calibrated beta, so the
accessibility measure and the demand model rest on one definition of a workplace
rather than two that quietly disagree. And a scheme changes journeys, so the
opportunities belonging in the sum are the ones somebody has to travel to reach.
`workplace_population` fails that test exactly where it differs from `jobs`, home
and no-fixed-place workers being the workers whose travel no scheme can alter;
that it barely moves the ranking is a reason to distrust the pair rather than a
reason to relax. `retail_jobs` answers a real question but a different one, and
here it is close to a map of one shopping centre. `population` measures access to
people, which is what you want for a firm's labour catchment and wrong for a
scheme about workers reaching jobs.

Where the argument is weakest. The upper reaches agree: under `jobs`,
`retail_jobs` and `workplace_population` the same five zones fill the top five,
and Chopwell & High Spen is last under all four. If the only decision resting on
these numbers is which corner of the conurbation is worst connected, the
opportunity variable hardly matters. Movement concentrates in the middle of the
distribution, where values sit within a few per cent of one another and small
zones with one dominant land use swing furthest. That limitation is also the
reason the choice has gone unexamined for so long. For the coarsest question
anyone asks it genuinely does not matter, so nobody acquires the habit of
checking before the question gets finer.